# 3 · Why did the price move?

The question historical data cannot answer.

You can observe that a stock fell. You cannot observe that 60% of the fall
was order-flow pressure and the rest was noise, unless something computed
it, which is the case here.

In [1]:
import struct
import pretium as pt

print("the seven factors:")
for f in pt.Engine.FACTORS:
    print("  ", f)

the seven factors:
   reversion
   momentum
   crowd_lean
   company_news
   order_flow_impact
   short_squeeze_effect
   random_noise


Four are shocks: `company_news`, `order_flow_impact`,
`short_squeeze_effect` and `random_noise`. Three are the model's own
dynamics: `reversion`, `momentum` and `crowd_lean`.

They accumulate per day and reset at market open, so reading after the close
gives you the day just finished.

In [2]:
universe = pt.Universe.random(20, seed=111)
engine = pt.Engine(universe=universe, seed=3)
engine.run_days(5)

def attribution_table(engine):
    cols = {}
    for factor in pt.Engine.FACTORS:
        raw = engine.attribution(factor)
        cols[factor] = struct.unpack(f"<{len(raw) // 8}d", raw)
    rows = []
    for i, ticker in enumerate(engine.tickers):
        parts = {f: cols[f][i] for f in pt.Engine.FACTORS}
        rows.append({"ticker": ticker, "total": sum(parts.values()), **parts})
    return rows

rows = attribution_table(engine)
rows.sort(key=lambda r: -abs(r["total"]))
print(f"{len(rows)} instruments, biggest movers on day 5 first")

20 instruments, biggest movers on day 5 first


## Checking the factors sum

Worth checking rather than taking on trust:

In [3]:
for r in rows[:3]:
    print(f"\n{r['ticker']}   total log move {r['total']:+.8f}")
    for f in pt.Engine.FACTORS:
        share = r[f] / r["total"] * 100 if r["total"] else 0.0
        bar = "#" * int(abs(share) / 4)
        print(f"    {f:22s} {r[f]:+.8f}  {share:6.1f}%  {bar}")
    residual = abs(sum(r[f] for f in pt.Engine.FACTORS) - r["total"])
    print(f"    {'residual':22s} {residual:.2e}")


AAP   total log move -0.05379755
    reversion              -0.00236714     4.4%  #
    momentum               -0.00641090    11.9%  ##
    crowd_lean             -0.00295730     5.5%  #
    company_news           +0.00000000    -0.0%  
    order_flow_impact      +0.00000000    -0.0%  
    short_squeeze_effect   -0.00800000    14.9%  ###
    random_noise           -0.03406222    63.3%  ###############
    residual               0.00e+00

AAG   total log move +0.04099604
    reversion              +0.00225980     5.5%  #
    momentum               +0.00193748     4.7%  #
    crowd_lean             +0.00169588     4.1%  #
    company_news           +0.00000000     0.0%  
    order_flow_impact      +0.00000000     0.0%  
    short_squeeze_effect   +0.00000000     0.0%  
    random_noise           +0.03510289    85.6%  #####################
    residual               0.00e+00

AAN   total log move -0.02184209
    reversion              +0.00057472    -2.6%  
    momentum               +0.

The residual is floating-point dust, around 1e-16. This is the engine's own
bookkeeping rather than a decomposition fitted afterwards.

## Across the whole roster

Aggregating the absolute contributions shows which mechanism is driving the
market on this day.

In [4]:
totals = {f: sum(abs(r[f]) for r in rows) for f in pt.Engine.FACTORS}
grand = sum(totals.values())

for f, v in sorted(totals.items(), key=lambda kv: -kv[1]):
    share = v / grand * 100
    print(f"  {f:22s} {share:5.1f}%  {'#' * int(share / 2)}")

  random_noise            63.6%  ###############################
  reversion               17.1%  ########
  crowd_lean              10.2%  #####
  momentum                 6.6%  ###
  short_squeeze_effect     2.5%  #
  company_news             0.0%  
  order_flow_impact        0.0%  


## Plotting it

A stacked view of the same data:

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

top = rows[:8]
fig, ax = plt.subplots(figsize=(10, 4.5))
bottom_pos = [0.0] * len(top)
bottom_neg = [0.0] * len(top)

for factor in pt.Engine.FACTORS:
    vals = [r[factor] for r in top]
    bases = [bottom_pos[i] if v >= 0 else bottom_neg[i] for i, v in enumerate(vals)]
    ax.bar([r["ticker"] for r in top], vals, bottom=bases, label=factor)
    for i, v in enumerate(vals):
        if v >= 0:
            bottom_pos[i] += v
        else:
            bottom_neg[i] += v

ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("contribution to log return")
ax.set_title("Who moved the price, day 5")
ax.legend(fontsize=7, ncol=4)
plt.tight_layout()
plt.show()

/var/folders/yw/6l1lnntd3vl29bv065bjtsnc0000gq/T/ipykernel_74730/3592016583.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## What this is for

No historical source can provide this labelling. If you are training a model
to attribute moves, or checking that a factor decomposition recovers what
actually happened, the ground truth here is exact.

Two caveats:

- This is the simulator's own bookkeeping. It is exact for this model and
  says nothing about why a real stock moved.
- No agent traded above, so `order_flow_impact` reflects background flow
  only. Run an evaluation to see a strategy's own footprint.

Next: **[4 · How realistic is this](04-how-realistic-is-this.ipynb)**.